# Gold Layer
Business ready aggregated tables

In [0]:
SILVER_PATH = "/Volumes/workspace/default/raw_data/silver"
GOLD_PATH   = "/Volumes/workspace/default/raw_data/gold"

In [0]:
# Loading Silver RFM feature table
customer_rfm = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/customer_rfm_features")

print(f"Silver RFM table loaded: {customer_rfm.count()} customers")
print(f"Columns: {customer_rfm.columns}")

In [0]:
print("CHURN LABEL DISTRIBUTION ")
customer_rfm.groupBy("is_churned").count().show()

print("RECENCY DISTRIBUTION (days) ")
customer_rfm.select("recency_days").describe().show()

print("MONETARY VALUE DISTRIBUTION ")
customer_rfm.select("monetary_value").describe().show()

In [0]:
from pyspark.sql.functions import col, when, round as spark_round

# Segment customers into churn risk buckets
gold_churn_segments = customer_rfm.withColumn(
    "churn_risk_segment",
    when(col("recency_days") <= 90, "Low Risk")
    .when(col("recency_days") <= 180, "Medium Risk")
    .when(col("recency_days") <= 365, "High Risk")
    .otherwise("Lost")
).withColumn(
    "rfm_segment",
    when((col("frequency") >= 2) & 
         (col("monetary_value") >= 200), "Champions")
    .when((col("frequency") >= 2) & 
          (col("monetary_value") < 200), "Loyal")
    .when((col("recency_days") <= 90) & 
          (col("frequency") == 1), "New Customer")
    .when((col("recency_days") > 180) & 
          (col("frequency") >= 2), "At Risk")
    .otherwise("Lost Customer")
).select(
    "customer_unique_id",
    "recency_days",
    "frequency",
    spark_round("monetary_value", 2).alias("monetary_value"),
    "avg_items_per_order",
    "avg_freight_value",
    "customer_state",
    "customer_city",
    "is_churned",
    "churn_risk_segment",
    "rfm_segment"
)

print("CHURN RISK SEGMENT DISTRIBUTION")
gold_churn_segments.groupBy("churn_risk_segment") \
    .count().orderBy("count", ascending=False).show()

print("RFM SEGMENT DISTRIBUTION")
gold_churn_segments.groupBy("rfm_segment") \
    .count().orderBy("count", ascending=False).show()

In [0]:
from pyspark.sql.functions import avg, count

print("AVG MONETARY VALUE BY RFM SEGMENT")
gold_churn_segments.groupBy("rfm_segment").agg(
    avg("monetary_value").alias("avg_spend"),
    avg("recency_days").alias("avg_recency"),
    avg("frequency").alias("avg_frequency"),
    count("customer_unique_id").alias("customer_count")
).orderBy("avg_spend", ascending=False).show()

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, count, round as spark_round

# Revenue and customer metrics aggregated by state
gold_revenue_by_state = customer_rfm.groupBy("customer_state").agg(
    count("customer_unique_id").alias("total_customers"),
    spark_round(spark_sum("monetary_value"), 2).alias("total_revenue"),
    spark_round(avg("monetary_value"), 2).alias("avg_customer_value"),
    spark_round(avg("recency_days"), 0).alias("avg_recency_days"),
    spark_sum("is_churned").alias("churned_customers")
).withColumn(
    "churn_rate_pct",
    spark_round((col("churned_customers") / col("total_customers")) * 100, 1)
).orderBy("total_revenue", ascending=False)

print("TOP 10 STATES BY REVENUE")
gold_revenue_by_state.show(10)

In [0]:
# ── Explore: Churn rate varies significantly by state? ──
print("HIGHEST CHURN RATE STATES (min 100 customers) ")
gold_revenue_by_state \
    .filter(col("total_customers") >= 100) \
    .orderBy("churn_rate_pct", ascending=False) \
    .show(10)

In [0]:
print("LOWEST CHURN RATE STATES")
gold_revenue_by_state \
    .filter(col("total_customers") >= 100) \
    .orderBy("churn_rate_pct", ascending=True) \
    .show(5)

In [0]:
from pyspark.sql.functions import date_format, to_timestamp

# Reload orders master for time series
BRONZE_PATH = "/Volumes/workspace/default/raw_data/bronze"
orders_b = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
payments_b = spark.read.format("delta").load(f"{BRONZE_PATH}/payments")

# Monthly revenue trend
payments_agg = payments_b.groupBy("order_id").agg(
    spark_sum("payment_value").alias("total_payment_value")
)

monthly_revenue = orders_b \
    .filter(col("order_status") == "delivered") \
    .join(payments_agg, "order_id", "left") \
    .withColumn("order_month", 
                date_format(to_timestamp(
                    col("order_purchase_timestamp")), "yyyy-MM")) \
    .groupBy("order_month").agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("total_payment_value"), 2).alias("monthly_revenue"),
        spark_round(avg("total_payment_value"), 2).alias("avg_order_value")
    ).orderBy("order_month")

print("MONTHLY REVENUE TREND ")
monthly_revenue.show(20)

In [0]:
# Finding peak revenue month
print("PEAK REVENUE MONTHS (TOP 5) ")
monthly_revenue.orderBy("monthly_revenue", ascending=False).show(5)

In [0]:
print("GROWTH: FIRST 3 vs LAST 3 MONTHS")
monthly_revenue.filter(col("order_month") <= "2017-01").show(3)
monthly_revenue.filter(col("order_month") >= "2018-06").show(3)

In [0]:
# write all 3 Gold tables as Delta
gold_churn_segments.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/customer_churn_segments")

gold_revenue_by_state.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/revenue_by_state")

monthly_revenue.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/monthly_revenue_trend")
print("Gold tables written to: ")
print(f"Location: {GOLD_PATH}")

In [0]:
# Gold tabel 1
print("GOLD TABLE 1: CUSTOMER CHURN SEGMENTS")
gold_1 = spark.read.format("delta") \
    .load(f"{GOLD_PATH}/customer_churn_segments")
print(f"Rows: {gold_1.count()}")
gold_1.groupBy("churn_risk_segment").count() \
    .orderBy("count", ascending=False).show()

In [0]:
# Gold table 2
print("GOLD TABLE 2: REVENUE BY STATE")
gold_2 = spark.read.format("delta") \
    .load(f"{GOLD_PATH}/revenue_by_state")
print(f"Rows: {gold_2.count()}")
gold_2.orderBy("total_revenue", ascending=False).show(5)

In [0]:
# Gold table 3
print("GOLD TABLE 3: MONTHLY REVENUE TREND")
gold_3 = spark.read.format("delta") \
    .load(f"{GOLD_PATH}/monthly_revenue_trend")
print(f"Rows: {gold_3.count()}")
gold_3.orderBy("monthly_revenue", ascending=False).show(5)